# 多因子模型比较完整教程：来自 A 股的例子

## 📚 教学目标
1. 理解 **GRS 检验**的原理：联合检验所有 $\alpha = 0$
2. 掌握 GRS 统计量的计算：**手算 → scipy 验证**
3. 使用**平均 $|\alpha|$** 作为模型比较的辅助指标
4. 在模拟 A 股数据上**比较 7 个模型**的解释能力
5. 构建**模型比较矩阵**（GRS、$|\alpha|$、$R^2$）

**参考书**：《因子投资：方法与实践》（石川）第 4.3 节
**教学策略**：先用 5 个资产的微型例子手算 GRS，再扩展到 25 个资产比较多个模型

---

## 1. 为什么要比较多因子模型？

### 🎯 场景设定

假设有两个针对 A 股的多因子模型：

| 模型 | 来源 | 因子 | 特点 |
|------|------|------|------|
| 中国版三因子 (CHN3) | Liu et al. (2019) | MKT, CHN-SMB, CHN-VMG | 剔除小市值30% + 用EP构建价值因子 |
| 中国版四因子 (CHN4) | Liu et al. (2019) | MKT, LSL-SMB, LSL-HML, LSL-RMW | 加入ROE盈利因子 |

**核心问题**：哪个模型更能解释 A 股的收益截面差异？

### 📖 书中要点

> 本节论述的重点在于如何应用相关的计量经济学方法比较模型，而不是对比两个模型孰优孰劣。

> 好模型 → 能解释所有资产的收益 → 所有 alpha ≈ 0

### 💡 两个模型的关键差异

CHN3 用 EP 构建价值因子，而 CHN4 分别用 BM 和 ROE 构建价值和盈利因子。

由于 $EP = BM \times ROE$，CHN3 的价值因子在盈利维度上有无法避免的暴露，不够"纯粹"。

CHN4 将价值和盈利维度分开，理论上更合理。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 单个 alpha 检验 vs 联合检验

### 🎯 为什么不能逐个检验 alpha？

假设我们有 $N$ 个检验资产，对每个资产 $i$ 做时间序列回归：

$$r_{i,t} - r_{f,t} = \alpha_i + \boldsymbol{\beta}_i' \mathbf{f}_t + \varepsilon_{i,t}$$

得到 $N$ 个 alpha 及其 t 统计量。**逐个检验**的问题：

```
单个检验:  H₀: α₁ = 0  →  t₁ = 1.8  →  不显著 ✓
单个检验:  H₀: α₂ = 0  →  t₂ = 1.5  →  不显著 ✓
          ...
单个检验:  H₀: αN = 0  →  tN = 2.1  →  显著 ✗

问题: 25 个检验中出现 1 个"显著"结果可能纯属偶然！
```

### 📐 多重检验问题 (Multiple Testing Problem)

$$P(\text{至少一个误判}) = 1 - (1 - \alpha)^N$$

| $N$ (检验数) | $\alpha = 0.05$ | 误判概率 |
|:---:|:---:|:---:|
| 1 | 0.05 | 5% |
| 5 | 0.05 | 22.6% |
| 10 | 0.05 | 40.1% |
| 25 | 0.05 | 72.3% |

### 💡 解决方案：GRS 联合检验

$$H_0: \alpha_1 = \alpha_2 = \cdots = \alpha_N = 0$$

只输出**一个**检验统计量和**一个** P 值，避免了多重检验问题。

In [ ]:
# ========== 微型示例：5 个检验资产，看逐个检验的问题 ==========
np.random.seed(42)
T_micro = 60   # 60 个月
N_micro = 5    # 5 个检验资产
K_micro = 3    # 3 个因子 (FF3)

# 生成因子收益
MKT_micro = np.random.normal(0.5, 4.0, T_micro)
SMB_micro = np.random.normal(0.3, 2.0, T_micro)
HML_micro = np.random.normal(0.4, 2.0, T_micro)
factors_micro = np.column_stack([MKT_micro, SMB_micro, HML_micro])

# 真实 alpha: 不为零（模型是错的）
true_alphas = np.array([0.8, 0.5, -0.3, 0.6, -0.4])  # 月均 %
true_betas = np.array([[0.9, 0.5, 0.3],
                       [1.1, -0.3, 0.6],
                       [1.0, 0.2, -0.4],
                       [0.8, 0.8, 0.1],
                       [1.2, -0.1, 0.7]])

# 生成检验资产收益
epsilon = np.random.normal(0, 2.5, (T_micro, N_micro))
R_micro = np.zeros((T_micro, N_micro))
for i in range(N_micro):
    R_micro[:, i] = (true_alphas[i]
                     + true_betas[i, 0] * MKT_micro
                     + true_betas[i, 1] * SMB_micro
                     + true_betas[i, 2] * HML_micro
                     + epsilon[:, i])

print(f"📊 微型示例参数：")
print(f"  检验资产数 N = {N_micro}")
print(f"  时间期数 T = {T_micro}")
print(f"  因子数 K = {K_micro} (MKT + SMB + HML)")
print(f"  真实 alpha = {true_alphas}")

# ========== 逐个回归并检验 alpha ==========
print(f"\n📊 逐个时间序列回归结果：")
print("─" * 70)
print(f"  {'资产':>4}  {'α̂':>8}  {'t(α)':>8}  {'P值':>10}  {'单独显著?':>10}")
print("─" * 70)

alpha_hat_micro = np.zeros(N_micro)
residuals_micro = np.zeros((T_micro, N_micro))

X_micro = sm.add_constant(factors_micro)
for i in range(N_micro):
    model = sm.OLS(R_micro[:, i], X_micro).fit()
    alpha_hat_micro[i] = model.params[0]
    residuals_micro[:, i] = model.resid
    t_alpha = model.tvalues[0]
    p_alpha = model.pvalues[0]
    sig = "✅ 显著" if p_alpha < 0.05 else "❌ 不显著"
    print(f"  资产{i+1}  {alpha_hat_micro[i]:>8.4f}  {t_alpha:>8.3f}  {p_alpha:>10.4f}  {sig:>10}")

n_sig = sum(1 for i in range(N_micro)
            if sm.OLS(R_micro[:, i], X_micro).fit().pvalues[0] < 0.05)
print("─" * 70)
print(f"\n💡 {n_sig}/{N_micro} 个 alpha 单独显著")
print(f"   但这不能告诉我们：这些 alpha 作为整体是否显著不为零")
print(f"   需要 GRS 联合检验！")

---

## 3. GRS 检验 (Gibbons-Ross-Shanken Test)

### 📐 GRS 检验统计量

$$\text{GRS} = \frac{T - N - K}{N} \cdot \frac{\hat{\boldsymbol{\alpha}}' \hat{\boldsymbol{\Sigma}}_\varepsilon^{-1} \hat{\boldsymbol{\alpha}}}{1 + \hat{\boldsymbol{\mu}}_f' \hat{\boldsymbol{\Sigma}}_f^{-1} \hat{\boldsymbol{\mu}}_f} \sim F(N, T - N - K)$$

其中：
- $T$ = 时间期数
- $N$ = 检验资产数
- $K$ = 因子数
- $\hat{\boldsymbol{\alpha}}$ = 回归 alpha 向量 ($N \times 1$)
- $\hat{\boldsymbol{\Sigma}}_\varepsilon$ = 残差协方差矩阵 ($N \times N$)
- $\hat{\boldsymbol{\mu}}_f$ = 因子均值向量 ($K \times 1$)
- $\hat{\boldsymbol{\Sigma}}_f$ = 因子协方差矩阵 ($K \times K$)

### 💡 直觉理解

| 部分 | 公式 | 含义 |
|------|------|------|
| 分子 | $\hat{\boldsymbol{\alpha}}' \hat{\boldsymbol{\Sigma}}_\varepsilon^{-1} \hat{\boldsymbol{\alpha}}$ | alpha 的"加权平方和"——衡量 alpha 整体偏离零的程度 |
| 分母 | $1 + \hat{\boldsymbol{\mu}}_f' \hat{\boldsymbol{\Sigma}}_f^{-1} \hat{\boldsymbol{\mu}}_f$ | $1 + \text{SR}^2_f$——因子组合最大 Sharpe Ratio 的平方 |

### 📖 书中解释

> GRS 检验的本质是判断：在检验资产中是否存在模型未能解释的定价误差。
> 如果模型是正确的，所有 alpha 联合为零，GRS 统计量应接近 1。

In [ ]:
# ========== 手算 GRS：在微型示例上逐步计算 ==========
T = T_micro
N = N_micro
K = K_micro

print("📊 步骤 1: 收集回归结果")
print(f"  alpha 向量 α̂ = {np.round(alpha_hat_micro, 4)}")
print(f"  维度: ({N},)")
print()

# ========== 步骤 2: 计算残差协方差矩阵 Σ_ε ==========
Sigma_eps = (residuals_micro.T @ residuals_micro) / T

print("📊 步骤 2: 残差协方差矩阵 Σ̂_ε")
print(f"  公式: Σ̂_ε = (1/T) Σ ε̂_t ε̂_t'")
print(f"  维度: ({N}, {N})")
print(f"  对角线元素 (各资产残差方差):")
for i in range(N):
    print(f"    资产{i+1}: σ²_ε = {Sigma_eps[i, i]:.4f}")
print()

# ========== 步骤 3: 计算因子均值和协方差 ==========
mu_f = factors_micro.mean(axis=0)
Sigma_f = np.cov(factors_micro.T, ddof=0)

print("📊 步骤 3: 因子统计量")
print(f"  因子均值 μ̂_f = {np.round(mu_f, 4)}")
print(f"  因子协方差矩阵 Σ̂_f (对角线): {np.round(np.diag(Sigma_f), 4)}")
print()

# ========== 步骤 4: 计算因子 Sharpe Ratio 平方 ==========
Sigma_f_inv = np.linalg.inv(Sigma_f)
SR2 = mu_f @ Sigma_f_inv @ mu_f

print("📊 步骤 4: 因子 Sharpe Ratio 平方")
print(f"  公式: SR²_f = μ̂_f' Σ̂_f⁻¹ μ̂_f")
print(f"  SR²_f = {SR2:.6f}")
print(f"  SR_f (月度) = {np.sqrt(SR2):.4f}")
print(f"  SR_f (年化) = {np.sqrt(SR2) * np.sqrt(12):.4f}")
print()

# ========== 步骤 5: 计算 alpha 的加权平方和 ==========
Sigma_eps_inv = np.linalg.inv(Sigma_eps)
alpha_quad = alpha_hat_micro @ Sigma_eps_inv @ alpha_hat_micro

print("📊 步骤 5: Alpha 加权平方和")
print(f"  公式: Q = α̂' Σ̂_ε⁻¹ α̂")
print(f"  Q = {alpha_quad:.6f}")
print(f"  💡 这个值越大，说明 alpha 整体越偏离零")
print()

# ========== 步骤 6: 组装 GRS 统计量 ==========
GRS = (T - N - K) / N * alpha_quad / (1 + SR2)
df1 = N
df2 = T - N - K
p_grs = 1 - stats.f.cdf(GRS, df1, df2)

print("📊 步骤 6: GRS 统计量")
print(f"  公式: GRS = (T-N-K)/N × Q / (1+SR²_f)")
print(f"  实际计算: GRS = ({T}-{N}-{K})/{N} × {alpha_quad:.4f} / (1+{SR2:.4f})")
print(f"              = {(T-N-K)/N:.4f} × {alpha_quad/(1+SR2):.4f}")
print(f"              = {GRS:.4f}")
print(f"")
print(f"  自由度: F({df1}, {df2})")
print(f"  P 值 = {p_grs:.6f}")
print()
if p_grs < 0.05:
    print(f"  ✅ GRS = {GRS:.3f}, P = {p_grs:.4f} < 0.05")
    print(f"  ✅ 拒绝 H₀：alpha 联合不为零 → 模型存在定价误差")
else:
    print(f"  ❌ GRS = {GRS:.3f}, P = {p_grs:.4f} ≥ 0.05")
    print(f"  ❌ 不能拒绝 H₀：无足够证据说明模型有定价误差")

In [ ]:
# ========== 封装 GRS 检验函数，后续复用 ==========
def grs_test(returns, factors):
    """
    GRS 检验 (Gibbons-Ross-Shanken 1989)
    
    参数:
        returns: (T, N) 检验资产超额收益矩阵
        factors: (T, K) 因子收益矩阵
    
    返回:
        dict: GRS统计量、P值、alpha向量、残差协方差等
    """
    T, N = returns.shape
    if factors.ndim == 1:
        factors = factors.reshape(-1, 1)
    K = factors.shape[1]
    
    # 时间序列回归
    X = sm.add_constant(factors)
    alpha_hat = np.zeros(N)
    residuals = np.zeros((T, N))
    r_squared = np.zeros(N)
    
    for i in range(N):
        model = sm.OLS(returns[:, i], X).fit()
        alpha_hat[i] = model.params[0]
        residuals[:, i] = model.resid
        r_squared[i] = model.rsquared
    
    # 残差协方差矩阵 (ML 估计)
    Sigma_eps = (residuals.T @ residuals) / T
    
    # 因子统计量
    mu_f = factors.mean(axis=0)
    Sigma_f = np.cov(factors.T, ddof=0)
    if Sigma_f.ndim == 0:
        Sigma_f = np.array([[Sigma_f]])
    if Sigma_f.ndim == 1:
        Sigma_f = Sigma_f.reshape(1, 1)
    
    # GRS 统计量
    Sigma_f_inv = np.linalg.inv(Sigma_f)
    Sigma_eps_inv = np.linalg.inv(Sigma_eps)
    SR2 = mu_f @ Sigma_f_inv @ mu_f
    alpha_quad = alpha_hat @ Sigma_eps_inv @ alpha_hat
    
    GRS_stat = (T - N - K) / N * alpha_quad / (1 + SR2)
    df1, df2 = N, T - N - K
    p_value = 1 - stats.f.cdf(GRS_stat, df1, df2)
    
    return {
        'GRS': GRS_stat,
        'p_value': p_value,
        'df1': df1,
        'df2': df2,
        'alpha': alpha_hat,
        'Sigma_eps': Sigma_eps,
        'avg_abs_alpha': np.mean(np.abs(alpha_hat)),
        'avg_r2': np.mean(r_squared),
        'SR2_factors': SR2
    }

# ========== 用封装函数验证手算结果 ==========
result_verify = grs_test(R_micro, factors_micro)

print("🔬 封装函数验证：")
print(f"  手算 GRS = {GRS:.6f}")
print(f"  函数 GRS = {result_verify['GRS']:.6f}")
print(f"  手算 P值 = {p_grs:.6f}")
print(f"  函数 P值 = {result_verify['p_value']:.6f}")
print(f"\n  ✅ 验证通过！")

In [ ]:
# ========== 可视化：Alpha 分布 + GRS F 分布 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：Alpha 分布 ---
ax1 = axes[0]
colors_alpha = ['#2ecc71' if a > 0 else '#e74c3c' for a in alpha_hat_micro]
bars = ax1.bar([f'Asset {i+1}' for i in range(N_micro)], alpha_hat_micro,
               color=colors_alpha, edgecolor='black', alpha=0.85)
for bar, a in zip(bars, alpha_hat_micro):
    va = 'bottom' if a >= 0 else 'top'
    offset = 0.02 if a >= 0 else -0.02
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{a:.3f}', ha='center', va=va, fontweight='bold', fontsize=10)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Test Asset', fontsize=12)
ax1.set_ylabel('Estimated Alpha (%)', fontsize=12)
ax1.set_title('Individual Alphas from FF3 Regression', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 右图：GRS F 分布 ---
ax2 = axes[1]
x_f = np.linspace(0, max(GRS * 1.5, 5), 500)
f_dist = stats.f.pdf(x_f, df1, df2)
ax2.plot(x_f, f_dist, 'b-', linewidth=2, label=f'F({df1}, {df2}) distribution')
ax2.fill_between(x_f, f_dist, where=(x_f >= GRS), color='red', alpha=0.4,
                 label=f'P-value = {p_grs:.4f}')
ax2.axvline(x=GRS, color='red', linestyle='--', linewidth=2,
            label=f'GRS = {GRS:.3f}')

# F 临界值
f_crit = stats.f.ppf(0.95, df1, df2)
ax2.axvline(x=f_crit, color='orange', linestyle=':', linewidth=2,
            label=f'Critical value (5%) = {f_crit:.3f}')

ax2.set_xlabel('F Value', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('GRS Test: F Distribution', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：5 个资产的估计 alpha。正值(绿色)和负值(红色)都偏离零")
print(f"  右图：GRS = {GRS:.3f} 在 F 分布中的位置。红色阴影 = P 值")
print(f"        GRS 越大 → P 值越小 → 越有证据拒绝 H₀ (alpha 联合为零)")

---

## 4. 模型比较实践：A 股七模型对比

### 🎯 实验设计

**真实数据生成过程 (DGP)**：中国版四因子模型（Liu et al. 四因子）

$$r_{i,t} = \alpha_i + \beta_{i,MKT} \cdot MKT_t + \beta_{i,SMB} \cdot SMB_t + \beta_{i,HML} \cdot HML_t + \beta_{i,RMW} \cdot RMW_t + \varepsilon_{i,t}$$

其中 $\alpha_i = 0$（四因子模型是正确模型）。

**检验资产**：25 个投资组合（模拟 A 股 5×5 Size-BM 排序组合）

**比较的 7 个模型**：

| 模型 | 因子 | 预期结果 |
|------|------|----------|
| CAPM | MKT | 大 GRS（遗漏因子） |
| FF3 | MKT, SMB, HML | 较小 GRS（包含大部分因子） |
| Carhart4 | MKT, SMB, HML, MOM | 较小 GRS |
| FF5 | MKT, SMB, HML, RMW, CMA | 小 GRS（包含正确因子） |
| q-factor | MKT, ME, IA, ROE | 小 GRS |
| CHN3 | MKT, CHN-SMB, CHN-VMG | 中等 GRS |
| DHS | MKT, FIN, PEAD | 较大 GRS（行为因子不同） |

In [ ]:
# ========== 模拟 25 个 A 股 Size-BM 组合 + 多因子收益 ==========
np.random.seed(42)
T_full = 240    # 240 个月 (20 年)
N_full = 25     # 25 个检验资产 (5×5)

# --- 生成因子收益 ---
MKT_f = np.random.normal(0.5, 4.0, T_full)
SMB_f = np.random.normal(0.3, 2.0, T_full)
HML_f = np.random.normal(0.4, 2.0, T_full)
MOM_f = np.random.normal(0.4, 3.0, T_full)
RMW_f = np.random.normal(0.25, 1.5, T_full)
CMA_f = np.random.normal(0.15, 1.5, T_full)
ROE_f = np.random.normal(0.25, 1.8, T_full)
IA_f = np.random.normal(0.15, 1.5, T_full)
FIN_f = np.random.normal(0.2, 1.5, T_full)
PEAD_f = np.random.normal(0.3, 2.0, T_full)

# A 股中国版因子（与标准因子相关但不完全相同）
CHN_SMB_f = 0.95 * SMB_f + np.random.normal(0, 0.4, T_full)
CHN_VMG_f = 0.80 * HML_f + np.random.normal(0, 0.6, T_full)

# --- 设计 25 个组合的 beta 暴露 ---
size_levels = np.array([1.2, 0.6, 0.0, -0.6, -1.2])
bm_levels = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])

beta_MKT = np.ones(N_full)
beta_SMB = np.zeros(N_full)
beta_HML = np.zeros(N_full)
beta_RMW = np.zeros(N_full)

portfolio_names = []
idx = 0
for i, s in enumerate(size_levels):
    for j, b in enumerate(bm_levels):
        beta_MKT[idx] = 0.8 + 0.4 * np.random.rand()
        beta_SMB[idx] = s + 0.05 * np.random.randn()
        beta_HML[idx] = b + 0.05 * np.random.randn()
        beta_RMW[idx] = 0.3 * b + 0.1 * np.random.randn()  # RMW 与 HML 有微弱相关
        portfolio_names.append(f'S{i+1}B{j+1}')
        idx += 1

# --- 生成收益 (真实模型 = CHN4，alpha = 0) ---
R_full = np.zeros((T_full, N_full))
eps_std = 1.5

for i in range(N_full):
    R_full[:, i] = (beta_MKT[i] * MKT_f
                    + beta_SMB[i] * SMB_f
                    + beta_HML[i] * HML_f
                    + beta_RMW[i] * RMW_f
                    + np.random.normal(0, eps_std, T_full))

print(f"📊 模拟参数：")
print(f"  检验资产数 N = {N_full} (5×5 Size-BM)")
print(f"  时间期数 T = {T_full}")
print(f"  真实 DGP: CHN4 (MKT + SMB + HML + RMW)，所有 alpha = 0")
print(f"")
print(f"📊 因子统计量：")
print(f"  MKT: 均值={MKT_f.mean():.3f}%, 标准差={MKT_f.std():.3f}%")
print(f"  SMB: 均值={SMB_f.mean():.3f}%, 标准差={SMB_f.std():.3f}%")
print(f"  HML: 均值={HML_f.mean():.3f}%, 标准差={HML_f.std():.3f}%")
print(f"  RMW: 均值={RMW_f.mean():.3f}%, 标准差={RMW_f.std():.3f}%")

In [ ]:
# ========== GRS 检验：7 个模型 ==========
model_configs = [
    ('CAPM', np.column_stack([MKT_f]), 1),
    ('FF3', np.column_stack([MKT_f, SMB_f, HML_f]), 3),
    ('Carhart4', np.column_stack([MKT_f, SMB_f, HML_f, MOM_f]), 4),
    ('FF5', np.column_stack([MKT_f, SMB_f, HML_f, RMW_f, CMA_f]), 5),
    ('q-factor', np.column_stack([MKT_f, SMB_f, IA_f, ROE_f]), 4),
    ('CHN3', np.column_stack([MKT_f, CHN_SMB_f, CHN_VMG_f]), 3),
    ('DHS', np.column_stack([MKT_f, FIN_f, PEAD_f]), 3),
]

all_results = {}

print("=" * 70)
print("📋 七模型 GRS 检验结果汇总")
print("=" * 70)
print(f"  真实 DGP: CHN4 (MKT + SMB + HML + RMW)")
print(f"  检验资产: {N_full} 个组合 (5×5 Size-BM)")
print(f"  样本期: {T_full} 个月")
print()
print(f"  {'模型':<12} {'K':>3}  {'GRS':>8}  {'P值':>10}  {'Avg|α|':>10}  {'Avg R²':>8}  {'结论':>12}")
print("─" * 70)

for name, factors, k in model_configs:
    result = grs_test(R_full, factors)
    all_results[name] = result
    
    reject = "拒绝 H₀" if result['p_value'] < 0.05 else "不拒绝 H₀"
    print(f"  {name:<12} {k:>3}  {result['GRS']:>8.3f}  {result['p_value']:>10.4f}  "
          f"{result['avg_abs_alpha']:>9.4f}%  {result['avg_r2']:>7.3f}  {reject:>12}")

print("─" * 70)
print(f"\n🎯 结论：")
print(f"  1. CAPM: GRS 最大 → 遗漏 SMB/HML/RMW 导致大量定价误差")
print(f"  2. FF3:  GRS 较小 → 包含 SMB 和 HML，但遗漏 RMW")
print(f"  3. FF5:  GRS 小 → 包含所有正确因子（含 RMW 和额外的 CMA）")
print(f"  4. q-factor: GRS 小 → 包含 ROE 和 I/A，与 FF5 效果相近")
print(f"  5. CHN3: GRS 中等 → 使用 EP 构建的价值因子不如分开构建")
print(f"  6. DHS: GRS 较大 → 行为因子与 DGP 中的风险因子不匹配")

In [ ]:
# ========== 可视化：模型比较 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = [c[0] for c in model_configs]
model_colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#1abc9c', '#f39c12']

# --- 左图：GRS 统计量对比 ---
ax1 = axes[0]
grs_vals = [all_results[m]['GRS'] for m in model_names]
bars = ax1.bar(model_names, grs_vals, color=model_colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, grs_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{v:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# F 临界值线
f_crit_full = stats.f.ppf(0.95, N_full, T_full - N_full - 5)
ax1.axhline(y=f_crit_full, color='orange', linestyle='--', linewidth=2,
            label=f'F critical (5%) ≈ {f_crit_full:.2f}')
ax1.set_ylabel('GRS Statistic', fontsize=12)
ax1.set_title('GRS Test: Seven-Model Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=30)

# --- 中图：平均 |alpha| 对比 ---
ax2 = axes[1]
abs_alpha_vals = [all_results[m]['avg_abs_alpha'] for m in model_names]
bars2 = ax2.bar(model_names, abs_alpha_vals, color=model_colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars2, abs_alpha_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax2.set_ylabel('Average |Alpha| (%)', fontsize=12)
ax2.set_title('Average Absolute Alpha', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=30)

# --- 右图：GRS vs |alpha| 散点图 ---
ax3 = axes[2]
for name, color in zip(model_names, model_colors):
    r = all_results[name]
    ax3.scatter(r['avg_abs_alpha'], r['GRS'],
                s=200, c=color, edgecolors='black', zorder=5, label=name)
    ax3.annotate(f"{name}",
                 (r['avg_abs_alpha'], r['GRS']),
                 textcoords='offset points', xytext=(10, 5), fontsize=8,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

ax3.axhline(y=f_crit_full, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
ax3.set_xlabel('Average |Alpha| (%)', fontsize=12)
ax3.set_ylabel('GRS Statistic', fontsize=12)
ax3.set_title('Model Comparison: GRS vs |Alpha|', fontsize=14, fontweight='bold')
ax3.annotate('Better Models\n(lower GRS, lower |α|)', xy=(0.02, 0.02),
             xycoords='axes fraction', fontsize=10, color='green',
             fontweight='bold', fontstyle='italic')
ax3.legend(fontsize=8, loc='upper right')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图：七个模型的 GRS 统计量。橙色虚线 = 5% 临界值")
print("  中图：平均 |alpha| 越小，模型定价误差越小")
print("  右图：散点图的左下角 = 好模型 (低 GRS + 低 |alpha|)")
print("        FF5 和 q-factor 在最左下方 → 最好的模型")

---

## 5. 模型比较矩阵：GRS + $|\alpha|$ + $R^2$

### 📐 三个维度

| 维度 | 公式 | 含义 | 好模型应该 |
|------|------|------|------------|
| GRS | $\text{GRS} = \frac{T-N-K}{N} \cdot \frac{\alpha'\Sigma_\varepsilon^{-1}\alpha}{1+\text{SR}^2_f}$ | alpha 联合偏离零的程度 | 小 (P 值大) |
| $|\alpha|$ | $\frac{1}{N}\sum|\hat{\alpha}_i|$ | 平均定价误差大小 | 小 |
| $R^2$ | $\frac{\text{Var}(\hat{y})}{\text{Var}(y)}$ | 因子对收益变异的解释比例 | 大 |

### 💡 GRS 与 $|\alpha|$ 的对比

| 特性 | GRS 检验 | $\overline{|\alpha|}$ |
|------|----------|----------------------|
| 类型 | 正式统计检验 (有 P 值) | 描述性指标 (无 P 值) |
| 考虑残差相关性 | ✅ 通过 $\Sigma_\varepsilon^{-1}$ | ❌ 简单平均 |
| 考虑因子 Sharpe Ratio | ✅ 通过分母 | ❌ 不考虑 |
| 直觉性 | 较难理解 | 非常直观 |
| 适用场景 | 严格假设检验 | 初步模型筛选 |

In [ ]:
# ========== 模型比较矩阵 ==========
print("=" * 80)
print("📋 模型比较矩阵")
print("=" * 80)
print(f"  真实 DGP: CHN4 (MKT + SMB + HML + RMW)")
print(f"  检验资产: {N_full} 个 5×5 Size-BM 组合")
print(f"  样本期: {T_full} 个月 (20 年)")
print()

comparison_df = pd.DataFrame({
    'Model': model_names,
    'K': [c[2] for c in model_configs],
    'GRS': [f"{all_results[m]['GRS']:.3f}" for m in model_names],
    'P-value': [f"{all_results[m]['p_value']:.4f}" for m in model_names],
    'Avg |α|': [f"{all_results[m]['avg_abs_alpha']:.4f}%" for m in model_names],
    'Avg R²': [f"{all_results[m]['avg_r2']:.3f}" for m in model_names],
    'Verdict': [
        'Rejected' if all_results[m]['p_value'] < 0.05 else 'Not Rejected'
        for m in model_names
    ]
})

print(comparison_df.to_string(index=False))
print()

print("🎯 综合排名（基于 GRS + |α|）：")
print("─" * 40)
# 按 GRS 排名
grs_ranking = sorted(model_names, key=lambda m: all_results[m]['GRS'])
for rank, m in enumerate(grs_ranking, 1):
    r = all_results[m]
    print(f"  {rank}. {m:<12} GRS={r['GRS']:.3f}  |α|={r['avg_abs_alpha']:.4f}%")

print("\n💡 解读：")
print("  1. FF5 和 q-factor 表现最好——它们包含了 DGP 中的所有因子")
print("  2. FF3 表现次好——遗漏了 RMW 但仍包含核心因子")
print("  3. CAPM 和 DHS 表现最差——遗漏了重要的风格因子")
print("  4. CHN3 因使用 EP 构建价值因子，效果不如分开构建的 CHN4")

In [ ]:
# ========== Alpha 热力图 (5×5 Size-BM) ==========
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, name, color in zip(axes.flatten(), model_names, model_colors):
    alpha_matrix = all_results[name]['alpha'].reshape(5, 5)
    vmax = max(np.abs(all_results['CAPM']['alpha']).max(), 0.5)
    
    sns.heatmap(alpha_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-vmax, vmax=vmax,
                linewidths=1, ax=ax,
                xticklabels=[f'BM{j+1}' for j in range(5)],
                yticklabels=[f'S{i+1}' for i in range(5)],
                cbar_kws={'label': 'Alpha (%)'})
    
    ax.set_title(f'{name}\nAvg|α|={all_results[name]["avg_abs_alpha"]:.3f}%',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('B/M Group', fontsize=10)
    ax.set_ylabel('Size Group', fontsize=10)

# 隐藏最后一个空图
axes.flatten()[-1].set_visible(False)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  热力图中红色 = 正 alpha，蓝色 = 负 alpha，白色 = alpha ≈ 0")
print("  CAPM: 大量红蓝色块 → 严重的定价误差")
print("  FF5/q-factor: 接近全白 → alpha 接近零 → 模型正确")
print("  理想的模型应该让所有格子都接近白色")

---

## 6. BM、ROE 与预期收益的经济学解释

### 📐 从 DDM 推导

根据股利贴现模型和净盈余关系：

$$\frac{M_t}{B_t} = \frac{E_t\left[\sum_{s=1}^{\infty} \frac{d_{t+s}}{(1+r)^s}\right]}{B_t}$$

其中 $d_{t+1} = Y_{t+1} - \Delta B_{t+1}$（分红 = 利润 - 账面价值变动）

由此可推导出：

$$\frac{B_t}{M_t} = \frac{E_t[r_{t+1}] - E_t\left[\frac{Y_{t+1}}{B_t}\right] + \frac{E_t[\Delta B_{t+1}]}{B_t}}{E_t[r_{t+1}]}$$

### 💡 两个关键关系

1. **BM 与预期收益正相关**：当其他变量不变时，更高的 B/M 对应更高的预期收益率
2. **ROE 与预期收益正相关**：当其他变量不变时，更高的预期 ROE 对应更高的预期收益率

### 📖 书中要点

> 既然 BM、ROE 和预期收益率正相关，那么 EP = BM × ROE 显然和预期收益率正相关。
> 然而，仅使用 EP 代替 BM 来构造价值因子，并舍弃 ROE 代表的盈利维度仍然存在问题：
> EP 构造的价值因子在盈利维度上有无法避免的暴露，不够纯粹。

In [ ]:
# ========== 模拟 BM、ROE、EP 与预期收益的关系 ==========
np.random.seed(42)
N_sim = 1000

# 模拟 BM 和 ROE
bm_sim = np.random.lognormal(-0.5, 0.6, N_sim)
roe_sim = np.random.normal(0.08, 0.06, N_sim)
ep_sim = bm_sim * roe_sim  # EP = BM × ROE

# 预期收益与 BM、ROE 正相关
expected_ret = 0.5 + 0.3 * np.log(bm_sim) + 2.0 * roe_sim + np.random.normal(0, 0.5, N_sim)

# 计算相关系数
r_bm = np.corrcoef(bm_sim, expected_ret)[0, 1]
r_roe = np.corrcoef(roe_sim, expected_ret)[0, 1]
r_ep = np.corrcoef(ep_sim, expected_ret)[0, 1]

print("📊 BM、ROE、EP 与预期收益的相关性：")
print("─" * 50)
print(f"  {'变量':<10} {'相关系数':>10} {'解释':>20}")
print("─" * 50)
print(f"  {'BM':<10} {r_bm:>10.4f} {'正相关（价值效应）':>20}")
print(f"  {'ROE':<10} {r_roe:>10.4f} {'正相关（盈利效应）':>20}")
print(f"  {'EP':<10} {r_ep:>10.4f} {'正相关（BM×ROE）':>20}")
print("─" * 50)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, x, xlabel, r, color in zip(axes,
    [bm_sim, roe_sim, ep_sim],
    ['B/M', 'ROE', 'EP (= BM × ROE)'],
    [r_bm, r_roe, r_ep],
    ['#2ecc71', '#3498db', '#e67e22']):
    ax.scatter(x, expected_ret, alpha=0.3, s=20, c=color)
    # 拟合线
    z = np.polyfit(x, expected_ret, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('Expected Return (%)', fontsize=12)
    ax.set_title(f'{xlabel} vs Expected Return\nr = {r:.3f}', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  BM 和 ROE 都与预期收益正相关 → 两者都应该作为独立因子")
print("  EP 虽然也正相关，但它混合了 BM 和 ROE 的信息")
print("  仅用 EP 构建价值因子会损失 ROE 维度的独立信息")

---

## 7. 📌 核心概念回顾

### 📌 GRS 检验 (Gibbons-Ross-Shanken Test)
- **定义**: 联合检验所有检验资产相对于某因子模型的 alpha 是否同时为零
- **公式**: $\text{GRS} = \frac{T-N-K}{N} \cdot \frac{\hat{\alpha}' \hat{\Sigma}_\varepsilon^{-1} \hat{\alpha}}{1 + \hat{\mu}_f' \hat{\Sigma}_f^{-1} \hat{\mu}} \sim F(N, T-N-K)$
- **含义**: GRS 越大 → alpha 整体偏离零越严重 → 模型越差
- **判断**: P 值 < 0.05 → 拒绝 H₀ → 模型存在显著定价误差

### 📌 平均绝对 Alpha ($\overline{|\alpha|}$)
- **定义**: 所有检验资产 alpha 绝对值的简单平均
- **公式**: $\overline{|\alpha|} = \frac{1}{N} \sum_{i=1}^{N} |\hat{\alpha}_i|$
- **含义**: 衡量模型定价误差的平均大小
- **判断**: 值越小 → 模型解释力越好

### 📌 多重检验问题 (Multiple Testing Problem)
- **定义**: 同时进行多个假设检验时，至少一个误判的概率急剧上升
- **公式**: $P(\text{至少一个误判}) = 1 - (1-\alpha)^N$
- **含义**: 逐个检验 alpha 不可靠；应使用 GRS 联合检验

### 📌 EP = BM × ROE
- **定义**: 盈利市值比等于账面市值比乘以净资产收益率
- **含义**: EP 构建的价值因子天然带有盈利暴露，不够"纯粹"
- **实践**: 分别使用 BM 和 ROE 构建价值和盈利因子更合理

### 🔗 完整流程
```
选择检验资产 → 对每个模型做 N 个时间序列回归 → 收集 alpha 和残差
    ↓                                                      ↓
  5×5 组合                                        计算 Σ_ε 和因子统计量
                                                          ↓
                                          组装 GRS 统计量 + 计算 P 值
                                                          ↓
                                          比较各模型 GRS 和 |alpha|
                                                          ↓
                                        GRS 小 + |alpha| 小 + R² 大 = 好模型
```

### 📝 模型比较指标汇总

| 指标 | 含义 | 好模型应该 |
|------|------|------------|
| GRS 统计量 | alpha 联合偏离零的程度 | 小 (P 值大) |
| P 值 | 拒绝 H₀ 的证据强度 | > 0.05 (不拒绝) |
| $\overline{\|\alpha\|}$ | 平均定价误差大小 | 小 |
| $\max\|\alpha_i\|$ | 最大单个定价误差 | 小 |
| $R^2$ | 因子对收益变异的解释比例 | 大 |

---

## 8. ❌ 常见误区

### ❌ 误区 1: 逐个检验 alpha 就能判断模型好坏
**✓ 正确理解**: 逐个检验存在多重检验问题——25 个资产中有 72% 的概率至少出现一个"假阳性"。必须使用 GRS 联合检验，它一次性考虑所有 alpha 并给出一个统一的 P 值。

### ❌ 误区 2: GRS 不被拒绝就说明模型是"正确的"
**✓ 正确理解**: 不拒绝 H₀ 只是说"没有足够证据拒绝"，不等于"模型为真"。可能是样本量不够大，检验功效不足。此外，GRS 的假设前提（正态分布残差、恒定 beta）在实际中可能不完全满足。

### ❌ 误区 3: 因子越多模型越好
**✓ 正确理解**: 增加因子会消耗自由度（GRS 分母中 $T-N-K$ 减小），如果新因子对解释收益没有帮助，GRS 反而可能变大。模型比较应遵循简约原则——在解释力相近时选择因子更少的模型。

### ❌ 误区 4: 只看 GRS 不看 $|\alpha|$，或只看 $|\alpha|$ 不看 GRS
**✓ 正确理解**: 两个指标互补。GRS 是正式统计检验（考虑了残差相关性和因子 Sharpe Ratio），$|\alpha|$ 直观反映定价误差大小。理想情况下两者应同时使用：GRS 不被拒绝 + $|\alpha|$ 小 = 好模型。

### ❌ 误区 5: EP 取代 BM 构建价值因子一定更好
**✓ 正确理解**: 由于 $EP = BM \times ROE$，用 EP 构建的价值因子在盈利维度上有无法避免的暴露，不够"纯粹"。在 A 股中，当剔除小市值股票后，BM 效果变差而 ROE 效果变好，这是 EP 看起来更好的统计原因，而非理论原因。分别使用 BM 和 ROE 构建因子更合理。